# Aula 02: Arquitetura ANSI/SPARC, Independência de Dados, Catálogo do Sistema e Sublinguagens SQL
**Disciplina:** Banco de Dados  
**Curso:** Tecnologia em Telemática -- IFCE Campus Tauá  
**Professor:** Prof. Me. Reginaldo Pereira Fernandes  

---
## 🎯 Objetivos do Laboratório
1. Compreender a arquitetura de 3 níveis e os mapeamentos conceituais.
2. Inspecionar o **Catálogo do Sistema (Dicionário de Dados)** consultando metadados em tempo real (`sqlite_master`).
3. Praticar as sublinguagens do SQL: **DDL**, **DML**, **DQL** e **TCL** (*Savepoints* e *Rollback*).
4. Avaliar na prática o conceito de **Independência Lógica e Física de Dados**.

### 1. DDL: Criação do Esquema Conceitual em SQLite
Vamos definir a estrutura das tabelas para gerenciamento de ativos de backbone de telecomunicações (`roteadores_core` e `circuitos_fibra`).

In [1]:
import sqlite3

# 1. Conexão com banco de dados em memória
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# 2. DDL: Criando tabelas de infraestrutura com restrições de integridade
cursor.execute("""
CREATE TABLE roteadores_core (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    hostname TEXT NOT NULL UNIQUE,
    ip_gerencia TEXT NOT NULL UNIQUE,
    fabricante TEXT NOT NULL,
    modelo TEXT NOT NULL,
    capacidade_gbps INTEGER NOT NULL CHECK (capacidade_gbps > 0)
);
""")

cursor.execute("""
CREATE TABLE circuitos_fibra (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    codigo_circuito TEXT NOT NULL UNIQUE,
    roteador_origem_id INTEGER NOT NULL,
    interface_local TEXT NOT NULL,
    banda_mbps INTEGER NOT NULL,
    status_operacional TEXT NOT NULL DEFAULT 'DOWN',
    FOREIGN KEY (roteador_origem_id) REFERENCES roteadores_core(id)
);
""")

print("✅ [DDL] Tabelas \x27roteadores_core\x27 e \x27circuitos_fibra\x27 criadas com sucesso!")

### 2. Catálogo do Sistema: Introspecção de Metadados
O SGBD mantém um catálogo interno (dicionário de dados) contendo as definições de todos os objetos do banco (*metadados*).
No SQLite, essa tabela canônica chama-se `sqlite_master` (ou `sqlite_schema`).

In [2]:
# Consulta ao catálogo do sistema
query_catalogo = """
SELECT type, name, tbl_name, sql 
FROM sqlite_master 
WHERE type IN ('table', 'index') AND name NOT LIKE 'sqlite_%';
"""

print("[-] Metadados registrados no Dicionário de Dados do SGBD:")
for obj_type, name, tbl_name, sql_def in cursor.execute(query_catalogo).fetchall():
    print(f"\n-> Tipo: {obj_type.upper()} | Nome: {name} | Tabela: {tbl_name}")
    print(f"   DDL original armazenado no catálogo:\n   {sql_def.strip()}")

### 3. DML e DQL: Manipulação e Consulta de Dados
Inserimos roteadores e circuitos e executamos uma consulta relacional com `JOIN` e filtros.

In [3]:
# DML: Inserindo roteadores de backbone
roteadores = [
    ("TAU-CORE-01", "10.200.1.1", "Cisco", "ASR-9006", 400),
    ("TAU-EDGE-01", "10.200.1.2", "Juniper", "MX480", 240),
    ("TAU-DIST-01", "10.200.1.3", "Huawei", "NE40E", 100)
]
cursor.executemany("INSERT INTO roteadores_core (hostname, ip_gerencia, fabricante, modelo, capacidade_gbps) VALUES (?, ?, ?, ?, ?)", roteadores)

# DML: Inserindo circuitos de fibra óptica
circuitos = [
    ("CIRC-TAU-FOR-01", 1, "HundredGigE0/0/0/1", 100000, "UP"),
    ("CIRC-TAU-CRAT-01", 1, "TenGigE0/0/0/2", 10000, "UP"),
    ("CIRC-TAU-IGUA-01", 2, "xe-0/1/0", 10000, "UP")
]
cursor.executemany("INSERT INTO circuitos_fibra (codigo_circuito, roteador_origem_id, interface_local, banda_mbps, status_operacional) VALUES (?, ?, ?, ?, ?)", circuitos)
conn.commit()
print("✅ [DML] Dados inseridos com sucesso!")

# DQL: Consulta de infraestrutura com JOIN
query = """
SELECT c.codigo_circuito, r.hostname, r.ip_gerencia, c.interface_local, c.banda_mbps, c.status_operacional
FROM circuitos_fibra c
JOIN roteadores_core r ON c.roteador_origem_id = r.id
ORDER BY c.banda_mbps DESC;
"""

print(f"\n{'CIRCUITO':<18} | {'ROTEADOR':<12} | {'IP GERENCIA':<14} | {'INTERFACE':<20} | {'BANDA (M)':<9} | STATUS")
print("-" * 85)
for row in cursor.execute(query).fetchall():
    print(f"{row[0]:<18} | {row[1]:<12} | {row[2]:<14} | {row[3]:<20} | {row[4]:<9} | {row[5]}")

### 4. Independência de Dados (Evolução Lógica e Otimização Física)
- **Independência Lógica:** Adicionamos uma nova coluna (`snmp_community`) via DDL. As consultas e aplicações já existentes continuam funcionando sem quebrar.
- **Independência Física:** Criamos um índice em disco (`CREATE INDEX`) para acelerar pesquisas por IP. A estrutura lógica da tabela e as consultas SQL permanecem intactas.

In [4]:
# Independencia Logica: Alterando a tabela sem impactar as aplicacoes
cursor.execute("ALTER TABLE roteadores_core ADD COLUMN snmp_community TEXT DEFAULT 'public';")
print("✅ [DDL] Coluna \x27snmp_community\x27 adicionada com sucesso.")

# Independencia Fisica: Criando indice para otimizacao interna de busca
cursor.execute("CREATE INDEX idx_roteadores_ip ON roteadores_core(ip_gerencia);")
print("✅ [DDL] Indice \x27idx_roteadores_ip\x27 criado no nivel fisico.")

# Verificando o catalogo atualizado
cursor.execute("SELECT name, type, sql FROM sqlite_master WHERE name = 'idx_roteadores_ip';")
print("[-] Indice registrado no catalogo do sistema:", cursor.fetchone())

### 5. TCL: Controle Transacional (ACID, Savepoints e Rollback)
As transações garantem que modificações críticas no banco sejam atômicas. Vamos demonstrar o uso de `SAVEPOINT` e `ROLLBACK` para reverter alterações indevidas ou violações de integridade.

In [5]:
# 1. Definindo ponto de salvamento
cursor.execute("SAVEPOINT sp_lab02;")

# 2. Simulando uma alteracao operacional indevida
cursor.execute("UPDATE roteadores_core SET capacidade_gbps = 999 WHERE hostname = 'TAU-CORE-01';")
print("⚠️  [DML] Valor alterado temporariamente na memoria transacional.")

# 3. Revertendo a operacao com ROLLBACK TO
cursor.execute("ROLLBACK TO sp_lab02;")
print("✅ [TCL] Operacao revertida (ROLLBACK TO SAVEPOINT) com sucesso!")

cursor.execute("SELECT hostname, capacidade_gbps FROM roteadores_core WHERE hostname = 'TAU-CORE-01';")
print("[-] Estado integro restaurado:", cursor.fetchone())

# 4. Teste de violacao de restricao CHECK capturada pelo SGBD
try:
    cursor.execute("UPDATE roteadores_core SET capacidade_gbps = -10 WHERE hostname = 'TAU-CORE-01';")
except sqlite3.IntegrityError as err:
    print(f"⚠️  [IntegrityError] O SGBD impediu a violacao de restricao: {err}")
    cursor.execute("ROLLBACK TO sp_lab02;")

# 5. Liberando o savepoint e confirmando o estado final
cursor.execute("RELEASE SAVEPOINT sp_lab02;")
conn.commit()
print("✅ [TCL] Transacao finalizada com seguranca (COMMIT).")